# Workflow: Files, Schedules, Warnings

The practical plumbing: JSON round-trips, time formats, schedule awareness,
departure tuning, and how the SDK talks to you when something is off.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt

from tqrouting import TQRouter, VRPInstance, Customer, VehicleType, Location, TimeWindow
from tqrouting.matrix import EuclideanMatrixProvider

data_dir = Path("data")

# Uses the TQROUTING_LICENSE_KEY environment variable by default.
# Set it before running: export TQROUTING_LICENSE_KEY="key/..."
router = TQRouter()

## 1. Instances from JSON files

`VRPInstance.from_json()` / `.to_json()` round-trip the user-friendly format.
Here: `cvrp_100.json`, a 100-customer CVRP.

In [2]:
instance_x = VRPInstance.from_json(data_dir / "cvrp_100.json")
print(f"CVRP: {len(instance_x.customers)} customers")

sol_x = router.solve(instance_x, time_limit=5)
print(f"Status: {sol_x.solution_status}, routes: {len(sol_x.routes)}, "
      f"duration {sol_x.total_route_duration:.0f}")

out_dir = Path("output")
out_dir.mkdir(exist_ok=True)
instance_x.to_json(out_dir / "instance_roundtrip.json")
again = VRPInstance.from_json(out_dir / "instance_roundtrip.json")
print(f"Round-trip OK: {len(again.customers)} customers")

[tqrouting] Solving VRP instance (100 customers, 1 vehicle types)...


CVRP: 100 customers


[tqrouting] Solver finished in 5.0s (status=Feasible, 25 routes, duration 25836.9, 0 unvisited).


Status: Feasible, routes: 25, duration 25837
Round-trip OK: 100 customers


## 2. Schedule awareness & departure times

With time windows, every visit gets an **arrival window**
(`earliest_arrival` / `latest_arrival`) and every route a departure window.
`set_departure_time()` reschedules a route and recomputes ETAs.

In [3]:
instance_sched = VRPInstance(
    customers=[
        Customer(location=Location(x=10, y=20), delivery_load=5, service_time="00:10:00",
                 time_window=TimeWindow(start="08:00:00", end="10:00:00")),
        Customer(location=Location(x=30, y=50), delivery_load=8, service_time="00:15:00",
                 time_window=TimeWindow(start="09:00:00", end="12:00:00")),
        Customer(location=Location(x=50, y=30), delivery_load=6, service_time="00:05:00",
                 time_window=TimeWindow(start="10:00:00", end="14:00:00")),
    ],
    fleet=[
        VehicleType(start_location=Location(x=0, y=0), end_location=Location(x=0, y=0),
                    capacity_load=30, shift_start="07:00:00", shift_end="18:00:00",
                    n_vehicles=3),
    ],
)
instance_sched.compute_distance_matrix(EuclideanMatrixProvider())
instance_sched.compute_duration_matrix(EuclideanMatrixProvider())

solution_sched = router.solve(instance_sched, time_limit=3)
route = solution_sched.routes[0]
print(f"Departure: {route.route_start_time:.0f}s, "
      f"window [{route.route_earliest_departure:.0f}, {route.route_latest_departure:.0f}]")
for visit in route.customer_sequence:
    print(f"  customer {visit.customer_id}: ETA {visit.eta:.0f} "
          f"(arrival window [{visit.earliest_arrival:.0f}, {visit.latest_arrival:.0f}])")

[tqrouting] Computing distance matrix for 4 unique locations...


[tqrouting] Distance matrix ready (4×4, 0.0s).


[tqrouting] Computing duration matrix for 4 unique locations...


[tqrouting] Duration matrix ready (4×4, 0.0s).


[tqrouting] Solving VRP instance (3 customers, 1 vehicle types)...


[tqrouting] Solver finished in 3.0s (status=Feasible, 1 routes, duration 1945.0, 0 unvisited).


Departure: 34413s, window [25200, 35978]
  customer 0: ETA 34436 (arrival window [25222, 36000])
  customer 1: ETA 35072 (arrival window [29436, 43200])
  customer 2: ETA 36000 (arrival window [33328, 50400])


In [4]:
# Depart at the latest possible time instead:
solution_sched.set_departure_time(route.id, route.route_latest_departure)
print(f"After rescheduling to {route.route_start_time:.0f}s:")
for visit in route.customer_sequence:
    print(f"  customer {visit.customer_id}: ETA {visit.eta:.0f}")

After rescheduling to 35978s:
  customer 0: ETA 36000
  customer 1: ETA 36636
  customer 2: ETA 37564


## 3. Serialization & time formats

Solutions convert to dicts / JSON. When the input used `HH:MM:SS` strings,
the output follows the same format automatically.

In [5]:
import json

result_dict = solution_sched.to_dict()
print("Auto-detected format (matches input):")
print(json.dumps(result_dict["routes"][0]["customer_sequence"][0], indent=2))

out_dir = Path("output")
out_dir.mkdir(exist_ok=True)
solution_sched.to_json(out_dir / "solution_demo.json", time_format="HH:MM:SS")
from tqrouting import VRPSolution
loaded = VRPSolution.from_json(out_dir / "solution_demo.json")
print(f"\nLoaded: {loaded.solution_status}, {len(loaded.routes)} routes")

Auto-detected format (matches input):
{
  "customer_id": 0,
  "eta": "10:00:00",
  "start_service_time": "10:00:00",
  "end_service_time": "10:10:00",
  "waiting_time": "00:00:00",
  "earliest_arrival": "07:00:22",
  "latest_arrival": "10:00:00",
  "no_wait_arrival": "08:00:00"
}

Loaded: Feasible, 1 routes


## 4. Infeasibility

When no feasible solution exists, the solve still returns the best attempt —
with `solution_status="Infeasible"`, a `UserWarning`, and the violation list.

In [6]:
import warnings

tight = VRPInstance(
    customers=[
        Customer(location=Location(x=10, y=10), delivery_load=30),
        Customer(location=Location(x=20, y=20), delivery_load=30),
    ],
    fleet=[VehicleType(start_location=Location(x=0, y=0), capacity_load=40, n_vehicles=1)],
)
tight.compute_duration_matrix(EuclideanMatrixProvider())

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    sol_bad = router.solve(tight, time_limit=2)

print(f"Status: {sol_bad.solution_status}")
for w in caught:
    print(f"warning: {w.message}")
for v in sol_bad.constraint_violations:
    print(f"  violation: {v.constraint} on route {v.route_id} (excess {v.excess_value})")

[tqrouting] Computing duration matrix for 3 unique locations...


[tqrouting] Duration matrix ready (3×3, 0.0s).


[tqrouting] Solving VRP instance (2 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Infeasible, 1 routes, duration 28.3, 0 unvisited).


Status: Infeasible
  violation: capacity on route 0 (excess 20.0)


Remedies: add vehicles/capacity, relax windows, or mark constraints as soft
(see [objectives.ipynb](objectives.ipynb)).

## 5. Warnings you might meet

The SDK warns instead of silently ignoring parameters that cannot act.
The full catalogue lives in the Error Handling docs page.

In [7]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    # profit_weight on an instance without profits does nothing - and says so.
    router.solve(tight, profit_weight=2.0, time_limit=1)
for w in caught:
    print(f"warning: {w.message}")

[tqrouting] Solving VRP instance (2 customers, 1 vehicle types)...


[tqrouting] Solver finished in 1.0s (status=Infeasible, 1 routes, duration 28.3, 0 unvisited).


## 6. Logging

Standard Python logging: one line when the solve starts, one when it finishes
(status, route count, duration, unvisited count).

In [8]:
import logging

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
router.solve(instance_sched, time_limit=2)
logging.disable(logging.INFO)   # quiet again for the rest of the notebook

[tqrouting] Solving VRP instance (3 customers, 1 vehicle types)...


tqrouting.router: Solving VRP instance (3 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 1 routes, duration 1945.0, 0 unvisited).


tqrouting.router: Solver finished in 2.0s (status=Feasible, 1 routes, duration 1945.0, 0 unvisited).
